# Transformer Grad-CAM S3D 



In [3]:
import os
import cv2
import math
import json
import random
import shutil
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.video import s3d, S3D_Weights, r3d_18, R3D_18_Weights

from IPython.display import Video, display, HTML

try:
    from tqdm import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x





In [4]:
# ============================================================
# 1. Configuration
# ============================================================

@dataclass
class CFG:
    dataset_root: str = "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition"
    backbone_name: str = "s3d"  # "s3d" is I3D-like; "r3d_18" is a ResNet3D baseline.
    pretrained: bool = True
    num_classes: int = 101

    clip_len: int = 32
    frame_size: int = 224
    batch_size: int = 2
    num_workers: int = 2
    epochs: int = 10
    lr: float = 1e-4
    weight_decay: float = 1e-4

    # Transformer temporal gate
    transformer_layers: int = 2
    transformer_heads: int = 8
    transformer_dropout: float = 0.1
    temporal_hidden_dim: int = 512

    # Explanation constraints
    lambda_spatial_sparse: float = 0.03
    lambda_spatial_tv: float = 0.02
    lambda_temporal_entropy: float = 0.02
    lambda_temporal_smooth: float = 0.02
    lambda_temporal_coverage: float = 0.10
    lambda_motion_align: float = 0.10
    lambda_spatial_focus: float = 0.10
    lambda_input_fidelity: float = 0.35
    lambda_strong_causal: float = 0.25

    target_temporal_coverage: float = 0.35
    fidelity_margin: float = 0.45
    topk_temporal_ratio: float = 0.35
    topk_spatial_ratio: float = 0.30
    deletion_fill: str = "mean"  # "zero" or "mean"

    # Optical flow settings
    use_optical_flow: bool = True
    flow_size: int = 112
    flow_topk_ratio: float = 0.30

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    output_dir: str = "./xai_outputs_v4"


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def split_dir(cfg: CFG, split: str) -> str:
    return os.path.join(cfg.dataset_root, split)


# ============================================================
# 2. Dataset UCF101 Folder Loader
# ============================================================

class UCF101VideoFolderDataset(Dataset):
    """
    Folder layout:
        root_dir/class_name/video.avi
    """
    def __init__(
        self,
        root_dir: str,
        clip_len: int = 32,
        frame_size: int = 224,
        train: bool = False,
        class_to_idx: Optional[Dict[str, int]] = None,
        max_videos_per_class: Optional[int] = None,
    ):
        self.root_dir = root_dir
        self.clip_len = clip_len
        self.frame_size = frame_size
        self.train = train

        classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = class_to_idx if class_to_idx is not None else {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

        exts = (".avi", ".mp4", ".mov", ".mkv")
        self.samples = []
        for class_name in classes:
            class_path = os.path.join(root_dir, class_name)
            videos = sorted([f for f in os.listdir(class_path) if f.lower().endswith(exts)])
            if max_videos_per_class is not None:
                videos = videos[:max_videos_per_class]
            if class_name not in self.class_to_idx:
                continue
            for f in videos:
                self.samples.append((os.path.join(class_path, f), self.class_to_idx[class_name], class_name))

        self.normalize = transforms.Normalize(
            mean=[0.43216, 0.394666, 0.37645],
            std=[0.22803, 0.22145, 0.216989],
        )

    def __len__(self):
        return len(self.samples)

    def _read_video(self, path: str) -> List[np.ndarray]:
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        return frames

    def _sample_indices(self, n: int) -> np.ndarray:
        if n <= 0:
            return np.zeros(self.clip_len, dtype=np.int64)
        if n >= self.clip_len:
            if self.train:
                start = random.randint(0, max(0, n - self.clip_len))
                return np.arange(start, start + self.clip_len)
            return np.linspace(0, n - 1, self.clip_len).astype(np.int64)
        return np.linspace(0, n - 1, self.clip_len).astype(np.int64)

    def _preprocess(self, frames: List[np.ndarray]) -> Tuple[torch.Tensor, torch.Tensor]:
        idx = self._sample_indices(len(frames))
        sampled = []
        for i in idx:
            if len(frames) == 0:
                fr = np.zeros((self.frame_size, self.frame_size, 3), dtype=np.uint8)
            else:
                fr = frames[int(i)]
            fr = cv2.resize(fr, (self.frame_size, self.frame_size))
            if self.train and random.random() < 0.5:
                fr = cv2.flip(fr, 1)
            sampled.append(fr)

        arr = np.stack(sampled).astype(np.float32) / 255.0  # T,H,W,C

        raw = torch.from_numpy(arr).permute(3, 0, 1, 2).float()  # C,T,H,W

        x = raw.clone()
        for t in range(x.shape[1]):
            x[:, t] = self.normalize(x[:, t])

        return x, raw

    def __getitem__(self, idx: int):
        path, y, class_name = self.samples[idx]
        frames = self._read_video(path)
        x, raw = self._preprocess(frames)
        return {
            "video": x,
            "raw_video": raw,
            "label": torch.tensor(y, dtype=torch.long),
            "path": path,
            "class_name": class_name,
        }


def build_datasets(cfg: CFG):
    train_ds = UCF101VideoFolderDataset(split_dir(cfg, "train"), cfg.clip_len, cfg.frame_size, train=True)
    val_ds = UCF101VideoFolderDataset(split_dir(cfg, "val"), cfg.clip_len, cfg.frame_size, train=False, class_to_idx=train_ds.class_to_idx)
    test_ds = UCF101VideoFolderDataset(split_dir(cfg, "test"), cfg.clip_len, cfg.frame_size, train=False, class_to_idx=train_ds.class_to_idx)
    print(f"Train videos: {len(train_ds)} | Val videos: {len(val_ds)} | Test videos: {len(test_ds)}")
    print(f"Classes: {len(train_ds.class_to_idx)}")
    return train_ds, val_ds, test_ds


def build_loaders(cfg: CFG):
    train_ds, val_ds, test_ds = build_datasets(cfg)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


# ============================================================
# 3. Video denormalization and optical-flow motion targets
# ============================================================


def _safe_float(x, default=None):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default


def temporal_importance_from_heatmap(heatmap: np.ndarray) -> torch.Tensor:
    """
    Calcule l'importance temporelle à partir de la heatmap finale affichée.
    Cette importance est utilisée pour évaluer l'explication réellement visible.
    """
    hm = np.asarray(heatmap, dtype=np.float32)
    if hm.ndim != 3:
        raise ValueError("heatmap doit être de forme [T,H,W].")
    scores = hm.reshape(hm.shape[0], -1).mean(axis=1)
    scores = scores - scores.min()
    scores = scores / (scores.sum() + 1e-8)
    return torch.tensor(scores, dtype=torch.float32)


def compute_temporal_explanation_metrics(heatmap: np.ndarray, top_percentile: float = 75.0) -> Dict[str, float]:
    """Métriques temporelles calculées depuis l'explication finale."""
    scores = temporal_importance_from_heatmap(heatmap).numpy()
    T = len(scores)
    p = scores / (scores.sum() + 1e-8)
    entropy = float(-(p * np.log(p + 1e-8)).sum() / (np.log(T + 1e-8))) if T > 1 else 0.0
    concentration = float((p ** 2).sum())
    order = np.argsort(-p)
    cum = np.cumsum(p[order])
    top80 = int(np.searchsorted(cum, 0.80) + 1)
    threshold = np.percentile(scores, top_percentile)
    coverage = float((scores >= threshold).mean())
    return {
        "temporal_explanation_entropy_lower_is_more_selective": entropy,
        "temporal_explanation_concentration_higher_is_more_selective": concentration,
        "temporal_explanation_top80_frames_lower_is_more_selective": float(top80),
        "temporal_explanation_coverage_ratio": coverage,
    }


def gate_diagnostics(alpha: Optional[torch.Tensor] = None, beta: Optional[torch.Tensor] = None) -> Dict[str, Optional[float]]:
    """Diagnostic de la porte temporelle Transformer, sans l'utiliser comme seule source d'explication."""
    out = {
        "temporal_gate_alpha_entropy_diagnostic": None,
        "temporal_gate_alpha_concentration_diagnostic": None,
        "temporal_gate_beta_mean_diagnostic": None,
        "temporal_gate_beta_std_diagnostic": None,
    }
    if alpha is not None:
        a = alpha.detach().cpu().float().view(-1).numpy()
        a = np.maximum(a, 1e-8)
        a = a / (a.sum() + 1e-8)
        out["temporal_gate_alpha_entropy_diagnostic"] = float(-(a * np.log(a)).sum() / (np.log(len(a) + 1e-8))) if len(a) > 1 else 0.0
        out["temporal_gate_alpha_concentration_diagnostic"] = float((a ** 2).sum())
    if beta is not None:
        b = beta.detach().cpu().float().view(-1).numpy()
        out["temporal_gate_beta_mean_diagnostic"] = float(np.mean(b))
        out["temporal_gate_beta_std_diagnostic"] = float(np.std(b))
    return out


def _upsample_heatmap_to_video(heatmap: np.ndarray, target_size: Tuple[int, int, int], device: str) -> torch.Tensor:
    """Upsampling heatmap [T,H,W] vers [1,1,T,H,W]."""
    hm = torch.tensor(heatmap, dtype=torch.float32, device=device).unsqueeze(0).unsqueeze(0)
    hm = F.interpolate(hm, size=target_size, mode="trilinear", align_corners=False)
    hm = hm.clamp(min=0)
    hm = hm / (hm.max() + 1e-8)
    return hm


def _make_blur_video(video: torch.Tensor, kernel_size: int = 31) -> torch.Tensor:
    """Crée une version floutée de la vidéo pour la deletion/insertion."""
    B, C, T, H, W = video.shape
    frames = video.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
    # moyenne locale simple et robuste, compatible GPU
    pad = kernel_size // 2
    blurred = F.avg_pool2d(frames, kernel_size=kernel_size, stride=1, padding=pad)
    blurred = blurred.reshape(B, T, C, H, W).permute(0, 2, 1, 3, 4)
    return blurred


def _dilate_mask_3d(mask: torch.Tensor, dilation: int = 7) -> torch.Tensor:
    """Dilatation spatiale d'un masque [1,1,T,H,W]."""
    if dilation <= 1:
        return mask
    B, C, T, H, W = mask.shape
    frames = mask.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
    pad = dilation // 2
    frames = F.max_pool2d(frames, kernel_size=dilation, stride=1, padding=pad)
    return frames.reshape(B, T, C, H, W).permute(0, 2, 1, 3, 4)


@torch.no_grad()
def _class_probability(model: nn.Module, video: torch.Tensor, target_class: int) -> float:
    out = model(video, return_explanations=False)
    if isinstance(out, dict):
        out = out.get("logits", out)
    return float(torch.softmax(out, dim=1)[0, target_class].detach().cpu().item())


def spatiotemporal_deletion_insertion_score(
    model: nn.Module,
    video: torch.Tensor,
    target_class: int,
    heatmap: np.ndarray,
    steps: int = 8,
    max_top_ratio: float = 0.35,
    dilation: int = 9,
) -> Tuple[float, float, float, float, List[float], List[float]]:
    """
    Deletion/Insertion sur les régions spatio-temporelles importantes de la heatmap finale.
    - Deletion AUC plus bas = meilleure causalité.
    - Insertion AUC plus haut = meilleure suffisance.
    - Deletion drop plus haut = plus causal.
    """
    model.eval()
    B, C, T, H, W = video.shape
    assert B == 1
    hm = _upsample_heatmap_to_video(heatmap, target_size=(T, H, W), device=video.device)
    base = _make_blur_video(video)

    ratios = np.linspace(0.0, max_top_ratio, steps + 1)
    deletion_probs, insertion_probs = [], []
    original_prob = _class_probability(model, video, target_class)

    for r in ratios:
        if r <= 0:
            vd = video.clone()
            vi = base.clone()
        else:
            thr = torch.quantile(hm.view(-1), max(0.0, 1.0 - float(r)))
            mask = (hm >= thr).float()
            mask = _dilate_mask_3d(mask, dilation=dilation)
            vd = video * (1 - mask) + base * mask
            vi = base * (1 - mask) + video * mask

        deletion_probs.append(_class_probability(model, vd, target_class))
        insertion_probs.append(_class_probability(model, vi, target_class))

    deletion_auc = float(np.trapezoid(deletion_probs, ratios) / max(ratios[-1] - ratios[0], 1e-8))
    insertion_auc = float(np.trapezoid(insertion_probs, ratios) / max(ratios[-1] - ratios[0], 1e-8))
    deletion_drop = float(original_prob - deletion_probs[-1])
    insertion_gain = float(insertion_probs[-1] - insertion_probs[0])
    return deletion_auc, insertion_auc, deletion_drop, insertion_gain, deletion_probs, insertion_probs


@torch.no_grad()
def temporal_frame_deletion_insertion_score(
    model: nn.Module,
    video: torch.Tensor,
    target_class: int,
    temporal_scores: torch.Tensor,
    steps: int = 8,
) -> Tuple[float, float, List[float], List[float]]:
    """Diagnostic secondaire : suppression/insertion de frames entières."""
    model.eval()
    B, C, T, H, W = video.shape
    scores = temporal_scores.detach().float().to(video.device).view(-1)
    if scores.numel() != T:
        scores = F.interpolate(scores.view(1, 1, -1), size=T, mode="linear", align_corners=False).view(-1)
    order = torch.argsort(scores, descending=True)
    fill = video.mean(dim=(2, 3, 4), keepdim=True)
    deletion_probs, insertion_probs = [], []
    chunk = max(1, T // steps)
    for s in range(0, T + 1, chunk):
        idx = order[:s]
        vd = video.clone()
        vi = fill.expand_as(video).clone()
        if s > 0:
            vd[:, :, idx] = fill[:, :, 0:1]
            vi[:, :, idx] = video[:, :, idx]
        deletion_probs.append(_class_probability(model, vd, target_class))
        insertion_probs.append(_class_probability(model, vi, target_class))
    deletion_auc = float(np.trapezoid(deletion_probs) / max(1, len(deletion_probs) - 1))
    insertion_auc = float(np.trapezoid(insertion_probs) / max(1, len(insertion_probs) - 1))
    return deletion_auc, insertion_auc, deletion_probs, insertion_probs







def describe_transformer_dynamics(heatmap: np.ndarray, video: torch.Tensor, cfg) -> Dict[str, Any]:
    """
    Diagnostic dynamique pour Transformer Grad-CAM.
    Il utilise optical flow + heatmap finale. Ce n'est pas une tête conceptuelle TCAV.
    """
    with torch.no_grad():
        temporal_flow, spatial_flow, _ = compute_optical_flow_targets(
            video.unsqueeze(0).to(cfg.device),
            out_hw=(heatmap.shape[1], heatmap.shape[2]),
            flow_size=getattr(cfg, "flow_size", 112),
            topk_ratio=getattr(cfg, "flow_topk_ratio", 0.2),
        )
    temporal_scores = temporal_importance_from_heatmap(heatmap)
    flow = temporal_flow[0].detach().cpu().float()
    if flow.numel() != temporal_scores.numel():
        flow = F.interpolate(flow.view(1, 1, -1), size=temporal_scores.numel(), mode="linear", align_corners=False).view(-1)

    scores = temporal_scores.numpy()
    important = np.where(scores >= np.percentile(scores, 75))[0]
    if len(important) == 0:
        important = np.array([int(np.argmax(scores))])

    motion_score = float(flow[important].mean().item())
    if motion_score >= 0.065:
        motion_level = "fort"
    elif motion_score >= 0.035:
        motion_level = "moyen"
    else:
        motion_level = "faible"

    accel = torch.abs(torch.diff(flow)).numpy()
    if len(important) > 1 and len(accel) > 0:
        valid_idx = important[important < len(accel)]
        variation_score = float(accel[valid_idx].mean()) if len(valid_idx) else float(accel.mean())
    else:
        variation_score = float(accel.mean()) if len(accel) else 0.0
    if variation_score >= 0.020:
        variation_level = "élevée"
    elif variation_score >= 0.010:
        variation_level = "modérée"
    else:
        variation_level = "faible"

    # localisation dynamique : pourcentage d'importance spatialement concentrée
    spatial_focus = []
    for t in important:
        h = heatmap[int(t)]
        active = h >= np.percentile(h, 75)
        if active.sum() > 0:
            spatial_focus.append(1.0 - active.mean())
    localization_value = float(np.mean(spatial_focus) * 100.0) if spatial_focus else None
    if localization_value is None:
        localization_label = "indéterminé"
    elif localization_value >= 70:
        localization_label = "localisé"
    elif localization_value >= 45:
        localization_label = "mixte"
    else:
        localization_label = "global"

    # continuité
    binary = np.zeros_like(scores)
    binary[important] = 1
    switches = np.abs(np.diff(binary)).sum()
    if switches <= 2:
        continuity = "stable"
    elif switches <= 5:
        continuity = "modérée"
    else:
        continuity = "intermittente"

    return {
        "segment": f"frames {int(important.min())} à {int(important.max())}",
        "motion_intensity_label": motion_level,
        "motion_intensity_value": round(motion_score, 4),
        "localization_label": localization_label,
        "localization_value": None if localization_value is None else round(localization_value, 2),
        "variation_label": variation_level,
        "variation_value": round(variation_score, 4),
        "continuity": continuity,
        "peak_frame": int(np.argmax(scores)),
    }




def denormalize_video(x: torch.Tensor) -> np.ndarray:
    """
    x: [C,T,H,W] normalized or raw.
    returns uint8 [T,H,W,C]
    """
    x = x.detach().cpu()
    if x.min() < 0:
        mean = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
        std = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)
        x = x * std + mean
    x = x.clamp(0, 1)
    return (x.permute(1, 2, 3, 0).numpy() * 255).astype(np.uint8)


@torch.no_grad()
def compute_optical_flow_targets(
    video: torch.Tensor,
    out_hw: Optional[Tuple[int, int]] = None,
    flow_size: int = 112,
    topk_ratio: float = 0.30,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Computes optical-flow magnitude pseudo-targets.

    input video: [B,C,T,H,W], normalized or raw.
    returns:
        temporal_flow: [B,T] normalized flow energy.
        spatial_flow: [B,1,T,Hout,Wout] normalized map in [0,1].
        binary_focus: [B,1,T,Hout,Wout] top-k pseudo object/motion region.
    """
    B, C, T, H, W = video.shape
    Hout, Wout = out_hw if out_hw is not None else (H, W)

    temporal_list = []
    spatial_list = []

    for b in range(B):
        frames = denormalize_video(video[b])
        gray_seq = []
        for fr in frames:
            gray = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY)
            gray = cv2.resize(gray, (flow_size, flow_size))
            gray_seq.append(gray)

        mags = []
        if T <= 1:
            mags = [np.zeros((flow_size, flow_size), dtype=np.float32)]
        else:
            first_mag = None
            for t in range(T - 1):
                prev = gray_seq[t]
                nxt = gray_seq[t + 1]
                flow = cv2.calcOpticalFlowFarneback(
                    prev, nxt, None,
                    pyr_scale=0.5, levels=3, winsize=15,
                    iterations=3, poly_n=5, poly_sigma=1.2, flags=0
                )
                mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2).astype(np.float32)
                if first_mag is None:
                    first_mag = mag
                mags.append(mag)
            mags = [first_mag] + mags

        mags = np.stack(mags)  # T,h,w
        maps = []
        for t in range(T):
            m = mags[t]
            m = m / (m.max() + 1e-8)
            m = cv2.resize(m, (Wout, Hout))
            maps.append(m)
        maps = np.stack(maps).astype(np.float32)  # T,Hout,Wout

        temporal = maps.mean(axis=(1, 2))
        temporal = temporal / (temporal.sum() + 1e-8)

        temporal_list.append(temporal)
        spatial_list.append(maps)

    temporal_flow = torch.tensor(np.stack(temporal_list), dtype=torch.float32, device=video.device)
    spatial_flow = torch.tensor(np.stack(spatial_list), dtype=torch.float32, device=video.device).unsqueeze(1)

    flat = spatial_flow.flatten(2)
    k = max(1, int(flat.shape[-1] * topk_ratio))
    thresh = torch.topk(flat, k=k, dim=-1).values[..., -1].view(B, 1, 1, 1, 1)
    binary_focus = (spatial_flow >= thresh).float()

    return temporal_flow, spatial_flow, binary_focus


@torch.no_grad()
def compute_frame_diff_targets(
    video: torch.Tensor,
    out_hw: Optional[Tuple[int, int]] = None,
    topk_ratio: float = 0.30,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Fast fallback if optical flow is too slow.
    """
    B, C, T, H, W = video.shape
    Hout, Wout = out_hw if out_hw is not None else (H, W)
    raw = video.detach()
    if raw.min() < 0:
        mean = torch.tensor([0.43216, 0.394666, 0.37645], device=video.device).view(1, 3, 1, 1, 1)
        std = torch.tensor([0.22803, 0.22145, 0.216989], device=video.device).view(1, 3, 1, 1, 1)
        raw = (raw * std + mean).clamp(0, 1)
    diff = torch.abs(raw[:, :, 1:] - raw[:, :, :-1]).mean(dim=1, keepdim=True)
    first = diff[:, :, :1]
    maps = torch.cat([first, diff], dim=2)
    maps = F.interpolate(maps, size=(T, Hout, Wout), mode="trilinear", align_corners=False)
    maps = maps / (maps.amax(dim=(2, 3, 4), keepdim=True) + 1e-8)
    temporal = maps.mean(dim=(1, 3, 4))
    temporal = temporal / (temporal.sum(dim=1, keepdim=True) + 1e-8)

    flat = maps.flatten(2)
    k = max(1, int(flat.shape[-1] * topk_ratio))
    thresh = torch.topk(flat, k=k, dim=-1).values[..., -1].view(B, 1, 1, 1, 1)
    binary_focus = (maps >= thresh).float()
    return temporal, maps, binary_focus


def describe_dynamics_from_flow(temporal_attention: torch.Tensor, temporal_flow: torch.Tensor) -> str:
    """
    temporal_attention: [T] or [T_feature] normalized/sigmoid.
    temporal_flow: [T] normalized.
    """
    att = temporal_attention.detach().cpu().float()
    flow = temporal_flow.detach().cpu().float()

    if att.numel() != flow.numel():
        att = F.interpolate(att.view(1, 1, -1), size=flow.numel(), mode="linear", align_corners=False).view(-1)

    att_np = att.numpy()
    flow_np = flow.numpy()
    T = len(flow_np)

    important = np.where(att_np >= np.percentile(att_np, 75))[0]
    if len(important) == 0:
        important = np.array([int(np.argmax(att_np))])

    peak = int(np.argmax(att_np))
    span = f"Frames importantes approximatives: {int(important.min())} à {int(important.max())}, pic à la frame {peak}."

    flow_top = float(flow_np[important].mean())
    flow_all = float(flow_np.mean() + 1e-8)
    acceleration = np.abs(np.diff(flow_np))
    abruptness = float(acceleration.max()) if len(acceleration) else 0.0
    mean_acc = float(acceleration.mean() + 1e-8) if len(acceleration) else 0.0

    if flow_top > 1.35 * flow_all:
        dyn = "La décision est dominée par un segment à mouvement fort, ce qui indique une contribution de la vitesse ou de l'intensité de l'action."
    elif abruptness > 2.0 * mean_acc:
        dyn = "La décision est liée à une rupture dynamique, c'est-à-dire un changement brusque du mouvement entre deux instants."
    else:
        dyn = "La décision repose sur une dynamique continue et régulière plutôt que sur un changement brutal."

    return span + " | " + dyn


In [5]:
# ============================================================
# 4. Model: Spatial Gate + Temporal Transformer Gate
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, dim: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class SpatialGate3D(nn.Module):
    """
    Learns WHERE to look.
    Input: [B,C,T,H,W]
    Output: [B,1,T,H,W]
    """
    def __init__(self, in_channels: int):
        super().__init__()
        mid = max(in_channels // 8, 16)
        self.net = nn.Sequential(
            nn.Conv3d(in_channels, mid, kernel_size=1),
            nn.BatchNorm3d(mid),
            nn.ReLU(inplace=True),
            nn.Conv3d(mid, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


class TemporalTransformerGate(nn.Module):
    """
    Learns WHEN to look.
    Replaces LSTM by Transformer Encoder.
    Uses sigmoid beta for coverage, and normalized alpha for context aggregation.
    """
    def __init__(self, in_channels: int, model_dim: int = 512, n_heads: int = 8, n_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.proj = nn.Linear(in_channels, model_dim)
        self.pos = PositionalEncoding(model_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=n_heads,
            dim_feedforward=model_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.att_head = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, model_dim // 2),
            nn.GELU(),
            nn.Linear(model_dim // 2, 1),
        )
        self.out_dim = model_dim

    def forward(self, frame_features: torch.Tensor):
        """
        frame_features: [B,T,C]
        returns:
            context: [B,D]
            alpha: [B,T], sum=1, for weighted aggregation and Grad-CAM fusion
            beta: [B,T], sigmoid activation, for coverage and important frame detection
            seq: [B,T,D]
        """
        x = self.proj(frame_features)
        x = self.pos(x)
        seq = self.encoder(x)
        logits = self.att_head(seq).squeeze(-1)
        beta = torch.sigmoid(logits)
        alpha = beta / (beta.sum(dim=1, keepdim=True) + 1e-8)
        context = torch.sum(seq * alpha.unsqueeze(-1), dim=1)
        return context, alpha, beta, seq


class EnhancedAttentionGatedTransformerGradCAMModel(nn.Module):
    """
    Explainable-by-design video classifier.
    Backbone predicts action after gated spatio-temporal filtering.
    """
    def __init__(self, cfg: CFG, num_classes: int = 101):
        super().__init__()
        self.cfg = cfg
        self.backbone_name = cfg.backbone_name
        self.num_classes = num_classes

        if cfg.backbone_name == "s3d":
            weights = S3D_Weights.DEFAULT if cfg.pretrained else None
            base = s3d(weights=weights)
            self.features = base.features
            feat_dim = 1024
        elif cfg.backbone_name == "r3d_18":
            weights = R3D_18_Weights.DEFAULT if cfg.pretrained else None
            base = r3d_18(weights=weights)
            self.features = nn.Sequential(base.stem, base.layer1, base.layer2, base.layer3, base.layer4)
            feat_dim = 512
        else:
            raise ValueError("cfg.backbone_name must be 's3d' or 'r3d_18'")

        self.feat_dim = feat_dim
        self.spatial_gate = SpatialGate3D(feat_dim)
        self.temporal_gate = TemporalTransformerGate(
            in_channels=feat_dim,
            model_dim=cfg.temporal_hidden_dim,
            n_heads=cfg.transformer_heads,
            n_layers=cfg.transformer_layers,
            dropout=cfg.transformer_dropout,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.temporal_gate.out_dim, num_classes),
        )

    def forward(self, x: torch.Tensor, return_explanations: bool = True):
        feats = self.features(x)  # [B,C,T',H',W']
        spatial = self.spatial_gate(feats)
        gated = feats * spatial

        frame_features = gated.mean(dim=(3, 4)).permute(0, 2, 1)  # [B,T',C]
        context, alpha, beta, seq = self.temporal_gate(frame_features)
        logits = self.classifier(context)

        if not return_explanations:
            return logits

        return {
            "logits": logits,
            "features": feats,
            "gated_features": gated,
            "spatial_gate": spatial,
            "temporal_alpha": alpha,
            "temporal_beta": beta,
            "temporal_seq": seq,
        }


# ============================================================
# 5. Explanation constraints and mixed loss
# ============================================================

def total_variation_3d(mask: torch.Tensor) -> torch.Tensor:
    tv_t = torch.abs(mask[:, :, 1:] - mask[:, :, :-1]).mean() if mask.shape[2] > 1 else torch.tensor(0.0, device=mask.device)
    tv_h = torch.abs(mask[:, :, :, 1:] - mask[:, :, :, :-1]).mean()
    tv_w = torch.abs(mask[:, :, :, :, 1:] - mask[:, :, :, :, :-1]).mean()
    return tv_t + tv_h + tv_w


def temporal_entropy(alpha: torch.Tensor) -> torch.Tensor:
    ent = -(alpha * torch.log(alpha + 1e-8)).sum(dim=1)
    ent = ent / math.log(alpha.shape[1] + 1e-8)
    return ent.mean()


def temporal_smoothness(beta: torch.Tensor) -> torch.Tensor:
    if beta.shape[1] <= 1:
        return torch.tensor(0.0, device=beta.device)
    return torch.abs(beta[:, 1:] - beta[:, :-1]).mean()


def temporal_coverage_loss(beta: torch.Tensor, target_ratio: float) -> torch.Tensor:
    """
    Prevents too small attention segments.
    beta is sigmoid attention [0,1], not normalized.
    """
    coverage = beta.mean(dim=1)
    return ((coverage - target_ratio) ** 2).mean()


def motion_alignment_loss(alpha: torch.Tensor, temporal_flow: torch.Tensor) -> torch.Tensor:
    """
    Aligns temporal attention with optical-flow motion profile.
    """
    flow = F.interpolate(temporal_flow.unsqueeze(1), size=alpha.shape[1], mode="linear", align_corners=False).squeeze(1)
    flow = flow / (flow.sum(dim=1, keepdim=True) + 1e-8)
    return F.mse_loss(alpha, flow)


def soft_iou_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    pred, target: [B,1,T,H,W] in [0,1].
    """
    inter = (pred * target).sum(dim=(1, 2, 3, 4))
    union = pred.sum(dim=(1, 2, 3, 4)) + target.sum(dim=(1, 2, 3, 4)) - inter
    return (1.0 - (inter + 1e-6) / (union + 1e-6)).mean()


def build_spatiotemporal_mask(
    spatial_gate: torch.Tensor,
    beta: torch.Tensor,
    input_size: Tuple[int, int, int],
    topk_temporal_ratio: float,
    topk_spatial_ratio: float,
) -> torch.Tensor:
    """
    Builds aggressive input-level mask from learned spatial and temporal gates.
    spatial_gate: [B,1,Tf,Hf,Wf]
    beta: [B,Tf]
    input_size: (T,H,W)
    returns mask [B,1,T,H,W]
    """
    B, _, Tf, Hf, Wf = spatial_gate.shape
    T, H, W = input_size

    kt = max(1, int(Tf * topk_temporal_ratio))
    temp_idx = torch.topk(beta, k=kt, dim=1).indices
    temp_mask = torch.zeros_like(beta)
    temp_mask.scatter_(1, temp_idx, 1.0)
    temp_mask = temp_mask.view(B, 1, Tf, 1, 1)

    flat = spatial_gate.view(B, 1, Tf, Hf * Wf)
    ks = max(1, int(Hf * Wf * topk_spatial_ratio))
    sp_idx = torch.topk(flat, k=ks, dim=-1).indices
    sp_mask = torch.zeros_like(flat)
    sp_mask.scatter_(-1, sp_idx, 1.0)
    sp_mask = sp_mask.view(B, 1, Tf, Hf, Wf)

    mask_feat = temp_mask * sp_mask
    mask = F.interpolate(mask_feat, size=(T, H, W), mode="trilinear", align_corners=False)
    mask = (mask > 0.25).float()
    return mask


def input_level_fidelity_loss(
    model: nn.Module,
    video: torch.Tensor,
    logits: torch.Tensor,
    spatial_gate: torch.Tensor,
    beta: torch.Tensor,
    labels: torch.Tensor,
    cfg: CFG,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """
    Stronger fidelity:
        insertion: important region/frames alone should preserve clean distribution.
        deletion: removing important region/frames should suppress predicted/true class.
        strong causal: target confidence must drop by margin after deletion.
    """
    B, C, T, H, W = video.shape
    with torch.no_grad():
        clean_probs = torch.softmax(logits, dim=1)
        pred_idx = clean_probs.argmax(dim=1)
        target_idx = labels

    mask = build_spatiotemporal_mask(
        spatial_gate=spatial_gate.detach(),
        beta=beta.detach(),
        input_size=(T, H, W),
        topk_temporal_ratio=cfg.topk_temporal_ratio,
        topk_spatial_ratio=cfg.topk_spatial_ratio,
    )

    if cfg.deletion_fill == "mean":
        fill = video.mean(dim=(2, 3, 4), keepdim=True)
    else:
        fill = torch.zeros_like(video)

    deleted = video * (1.0 - mask) + fill * mask
    inserted = video * mask + fill * (1.0 - mask)

    logits_deleted = model(deleted, return_explanations=False)
    logits_inserted = model(inserted, return_explanations=False)

    # Insertion: selected evidence should reconstruct the clean decision distribution.
    insertion_bce = F.binary_cross_entropy_with_logits(logits_inserted, clean_probs.detach())

    # Deletion: selected evidence removed should not support the predicted class.
    deletion_target = clean_probs.detach().clone()
    deletion_target.scatter_(1, pred_idx.view(-1, 1), 0.0)
    deletion_bce = F.binary_cross_entropy_with_logits(logits_deleted, deletion_target)

    # Strong causal drop: confidence of the true class should drop after deletion.
    clean_target_prob = clean_probs.gather(1, target_idx.view(-1, 1)).squeeze(1).detach()
    deleted_target_prob = torch.softmax(logits_deleted, dim=1).gather(1, target_idx.view(-1, 1)).squeeze(1)
    strong_causal = F.relu(deleted_target_prob - clean_target_prob + cfg.fidelity_margin).mean()

    loss = insertion_bce + deletion_bce + cfg.lambda_strong_causal * strong_causal

    parts = {
        "input_insertion_bce": float(insertion_bce.detach().cpu()),
        "input_deletion_bce": float(deletion_bce.detach().cpu()),
        "strong_causal": float(strong_causal.detach().cpu()),
    }
    return loss, parts


def mixed_loss(outputs: Dict, labels: torch.Tensor, video: torch.Tensor, model: nn.Module, cfg: CFG):
    logits = outputs["logits"]
    spatial = outputs["spatial_gate"]
    alpha = outputs["temporal_alpha"]
    beta = outputs["temporal_beta"]

    pred_loss = F.cross_entropy(logits, labels)

    if cfg.use_optical_flow:
        temporal_flow, spatial_flow, binary_focus = compute_optical_flow_targets(
            video.detach(),
            out_hw=(spatial.shape[-2], spatial.shape[-1]),
            flow_size=cfg.flow_size,
            topk_ratio=cfg.flow_topk_ratio,
        )
    else:
        temporal_flow, spatial_flow, binary_focus = compute_frame_diff_targets(
            video.detach(),
            out_hw=(spatial.shape[-2], spatial.shape[-1]),
            topk_ratio=cfg.flow_topk_ratio,
        )

    # Match feature time dimension if needed.
    if binary_focus.shape[2] != spatial.shape[2]:
        binary_focus = F.interpolate(binary_focus, size=spatial.shape[2:], mode="trilinear", align_corners=False)
    if spatial_flow.shape[2] != spatial.shape[2]:
        spatial_flow = F.interpolate(spatial_flow, size=spatial.shape[2:], mode="trilinear", align_corners=False)

    spatial_sparse = spatial.mean()
    spatial_tv = total_variation_3d(spatial)
    temp_ent = temporal_entropy(alpha)
    temp_smooth = temporal_smoothness(beta)
    temp_cov = temporal_coverage_loss(beta, cfg.target_temporal_coverage)
    motion_align = motion_alignment_loss(alpha, temporal_flow)
    spatial_focus = soft_iou_loss(spatial, binary_focus)

    fidelity, fid_parts = input_level_fidelity_loss(
        model=model,
        video=video,
        logits=logits,
        spatial_gate=spatial,
        beta=beta,
        labels=labels,
        cfg=cfg,
    )

    total = (
        pred_loss
        + cfg.lambda_spatial_sparse * spatial_sparse
        + cfg.lambda_spatial_tv * spatial_tv
        + cfg.lambda_temporal_entropy * temp_ent
        + cfg.lambda_temporal_smooth * temp_smooth
        + cfg.lambda_temporal_coverage * temp_cov
        + cfg.lambda_motion_align * motion_align
        + cfg.lambda_spatial_focus * spatial_focus
        + cfg.lambda_input_fidelity * fidelity
    )

    logs = {
        "loss_total": float(total.detach().cpu()),
        "loss_pred": float(pred_loss.detach().cpu()),
        "spatial_sparse": float(spatial_sparse.detach().cpu()),
        "spatial_tv": float(spatial_tv.detach().cpu()),
        "temporal_entropy": float(temp_ent.detach().cpu()),
        "temporal_smooth": float(temp_smooth.detach().cpu()),
        "temporal_coverage": float(temp_cov.detach().cpu()),
        "motion_align_optical_flow": float(motion_align.detach().cpu()),
        "spatial_focus_iou_loss": float(spatial_focus.detach().cpu()),
        "input_fidelity": float(fidelity.detach().cpu()),
        **fid_parts,
    }
    return total, logs


# ============================================================
# 6. Grad-CAM fused with learned gates
# ============================================================

class VideoGradCAM:
    """Grad-CAM guidé pour la variation Transformer-Gated Grad-CAM."""
    def __init__(self, model: nn.Module):
        self.model = model

    def __call__(self, video: torch.Tensor, class_idx: Optional[int] = None):
        device = next(self.model.parameters()).device
        self.model.eval()
        video = video.clone().detach().to(device)
        video.requires_grad_(True)

        out = self.model(video, return_explanations=True)
        logits = out["logits"] if isinstance(out, dict) else out
        probs = torch.softmax(logits, dim=1)

        if class_idx is None:
            class_idx = int(probs.argmax(dim=1).item())

        score = logits[:, class_idx].sum()
        feats = out.get("gated_features", out.get("features", None))
        if feats is None:
            raise KeyError("Le modèle doit retourner 'gated_features' ou 'features' pour Grad-CAM.")

        self.model.zero_grad(set_to_none=True)
        grads = torch.autograd.grad(score, feats, retain_graph=True, create_graph=False)[0]
        weights = grads.mean(dim=(2, 3, 4), keepdim=True)
        cam = F.relu((weights * feats).sum(dim=1, keepdim=True))

        spatial = out.get("spatial_gate", None)
        if spatial is not None and spatial.shape[2:] == cam.shape[2:]:
            cam = cam * spatial.detach()

        alpha = out.get("temporal_alpha", None)
        beta = out.get("temporal_beta", None)
        if alpha is not None:
            a = alpha.detach()
            a_5d = F.interpolate(a.view(1, 1, -1), size=cam.shape[2], mode="linear", align_corners=False).view(1, 1, cam.shape[2], 1, 1)
            cam = cam * a_5d

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam_up = F.interpolate(cam, size=(video.shape[2], video.shape[3], video.shape[4]), mode="trilinear", align_corners=False)
        heatmap = cam_up[0, 0].detach().cpu().numpy()

        if beta is None and alpha is not None:
            beta = alpha.detach()
        if beta is None:
            beta = temporal_importance_from_heatmap(heatmap).view(1, -1).to(device)
        if alpha is None:
            alpha = beta

        return heatmap, alpha[0].detach().cpu(), beta[0].detach().cpu(), class_idx, float(probs[0, class_idx].detach().cpu())



# ============================================================
# 7. Visualization
# ============================================================

def overlay_heatmap(frame_rgb: np.ndarray, heat: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heat_uint = np.uint8(255 * np.clip(heat, 0, 1))
    color = cv2.applyColorMap(heat_uint, cv2.COLORMAP_JET)
    color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(frame_rgb, 1 - alpha, color, alpha, 0)


def save_explanation_video(
    video_tensor: torch.Tensor,
    heatmap: np.ndarray,
    temporal_beta: torch.Tensor,
    pred_text: str,
    prob: float,
    dynamics_text: str,
    save_path: str,
    fps: int = 8,
):
    frames = denormalize_video(video_tensor)
    T, H, W, _ = frames.shape

    beta = F.interpolate(temporal_beta.view(1, 1, -1), size=T, mode="linear", align_corners=False).view(-1).numpy()
    threshold = np.percentile(beta, 75)
    important = beta >= threshold

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(save_path, fourcc, fps, (W, H))

    for t in range(T):
        frame = overlay_heatmap(frames[t], heatmap[t])
        if important[t]:
            cv2.rectangle(frame, (3, 3), (W - 4, H - 4), (255, 0, 0), 5)

        txt1 = f"Prediction: {pred_text} ({prob:.2f})"
        txt2 = f"Temporal gate: {beta[t]:.3f}"
        cv2.putText(frame, txt1, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, txt2, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    writer.release()


def save_original_clip(sample, save_path: str, fps: int = 8):
    frames = denormalize_video(sample["video"])
    T, H, W, _ = frames.shape
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(save_path, fourcc, fps, (W, H))
    for fr in frames:
        writer.write(cv2.cvtColor(fr, cv2.COLOR_RGB2BGR))
    writer.release()
    return save_path


def convert_to_web_mp4(input_path: str, output_path: str):
    os.system(f'ffmpeg -y -i "{input_path}" -vcodec libx264 -pix_fmt yuv420p "{output_path}" -loglevel quiet')
    return output_path


def show_original_and_explanation(report, sample=None, width: int = 750):
    web_dir = "./xai_outputs_transformer_s3d/web"
    os.makedirs(web_dir, exist_ok=True)

    if sample is not None:
        original_path = os.path.join(web_dir, "original_tmp.mp4")
        save_original_clip(sample, original_path)
        original_web = os.path.join(web_dir, "original_web.mp4")
        convert_to_web_mp4(original_path, original_web)
        print("Vidéo originale")
        display(Video(original_web, embed=True, width=width))

    explained_web = os.path.join(web_dir, "explained_web.mp4")
    convert_to_web_mp4(report["output_video"], explained_web)
    print("Vidéo expliquée")
    display(Video(explained_web, embed=True, width=width))

    dyn = report.get("important_dynamics", {})
    display(HTML(f"""
    <div style="font-family: Arial; max-width: 980px; line-height: 1.45;">
      <h3>Explication complète — Transformer Grad-CAM</h3>
      <b>Ground truth:</b> {report.get('ground_truth')}<br>
      <b>Prediction:</b> {report.get('prediction')}<br>
      <b>Correct:</b> {report.get('correct')}<br>
      <b>Confidence:</b> {report.get('probability'):.4f}<br>

      <h4>Axes d'explication disponibles</h4>
      <ul>
        <li><b>Où:</b> heatmap Grad-CAM spatio-temporelle.</li>
        <li><b>Quand:</b> importance temporelle dérivée de la heatmap finale.</li>
        <li><b>Comment:</b> diagnostic dynamique par optical flow pondéré par l'explication.</li>
      </ul>

      <h4>Dynamique</h4>
      <ul>
        <li><b>Segment:</b> {dyn.get('segment')}</li>
        <li><b>Intensité:</b> {dyn.get('motion_intensity_label')} ({dyn.get('motion_intensity_value')})</li>
        <li><b>Localisation:</b> {dyn.get('localization_label')} ({dyn.get('localization_value')}%)</li>
        <li><b>Variation:</b> {dyn.get('variation_label')} ({dyn.get('variation_value')})</li>
        <li><b>Continuité:</b> {dyn.get('continuity')}</li>
      </ul>

      <h4>Métriques principales</h4>
      <ul>
        <li><b>Deletion AUC spatio-temporelle:</b> {report.get('spatiotemporal_deletion_auc_lower_is_better'):.4f}</li>
        <li><b>Insertion AUC spatio-temporelle:</b> {report.get('spatiotemporal_insertion_auc_higher_is_better'):.4f}</li>
        <li><b>Deletion drop final:</b> {report.get('spatiotemporal_deletion_drop_higher_is_better'):.4f}</li>
        <li><b>Stabilité:</b> {report.get('temporal_stability_lower_is_better'):.6f}</li>
        <li><b>Motion proxy:</b> {report.get('motion_localization_proxy_higher_is_better'):.4f}</li>
      </ul>

      <h4>Diagnostic temporel</h4>
      <ul>
        <li><b>Entropie explication:</b> {report.get('temporal_explanation_entropy_lower_is_more_selective'):.4f}</li>
        <li><b>Concentration explication:</b> {report.get('temporal_explanation_concentration_higher_is_more_selective'):.4f}</li>
        <li><b>Frames pour 80% importance:</b> {report.get('temporal_explanation_top80_frames_lower_is_more_selective'):.2f}</li>
      </ul>
    </div>
    """))


# ============================================================
# 8. Evaluation metrics
# ============================================================

def accuracy(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item()


def heatmap_temporal_stability(heatmap: np.ndarray) -> float:
    """Mesure simple du flickering entre heatmaps successives."""
    hm = np.asarray(heatmap, dtype=np.float32)
    if hm.shape[0] <= 1:
        return 0.0
    return float(np.mean(np.abs(np.diff(hm, axis=0))))


def localization_motion_proxy(heatmap: np.ndarray, video: torch.Tensor, cfg) -> float:
    """Alignement moyen heatmap/optical-flow. Diagnostic dynamique commun."""
    with torch.no_grad():
        temporal_flow, spatial_flow, _ = compute_optical_flow_targets(
            video.unsqueeze(0).to(cfg.device),
            out_hw=(heatmap.shape[1], heatmap.shape[2]),
            flow_size=getattr(cfg, "flow_size", 112),
            topk_ratio=getattr(cfg, "flow_topk_ratio", 0.2),
        )
    flow = spatial_flow[0, 0].detach().cpu().numpy()
    if flow.shape != heatmap.shape:
        flow_t = torch.tensor(flow).unsqueeze(0).unsqueeze(0).float()
        flow = F.interpolate(flow_t, size=heatmap.shape, mode="trilinear", align_corners=False)[0, 0].numpy()
    hm = heatmap / (heatmap.max() + 1e-8)
    fl = flow / (flow.max() + 1e-8)
    return float((hm * fl).sum() / (hm.sum() + 1e-8))



@torch.no_grad()
def temporal_deletion_insertion_score(
    model: nn.Module,
    video: torch.Tensor,
    label: torch.Tensor,
    beta: torch.Tensor,
    steps: int = 8,
):
    """
    Pixel-space temporal fidelity.
    Deletion: remove most important frames first. Lower AUC is better.
    Insertion: insert most important frames first. Higher AUC is better.
    """
    model.eval()
    B, C, T, H, W = video.shape
    assert B == 1

    label_int = int(label.item())
    beta_i = F.interpolate(beta.view(1, 1, -1).to(video.device), size=T, mode="linear", align_corners=False).view(-1)
    order = torch.argsort(beta_i, descending=True)

    if video.min() < 0:
        fill = video.mean(dim=(2, 3, 4), keepdim=True)
    else:
        fill = torch.zeros_like(video)

    deletion_probs, insertion_probs = [], []
    chunk = max(1, T // steps)

    for s in range(0, T + 1, chunk):
        idx = order[:s]
        vd = video.clone()
        vi = fill.clone().expand_as(video).clone()

        if s > 0:
            vd[:, :, idx] = fill[:, :, 0:1]
            vi[:, :, idx] = video[:, :, idx]

        pd = torch.softmax(model(vd, return_explanations=False), dim=1)[0, label_int].item()
        pi = torch.softmax(model(vi, return_explanations=False), dim=1)[0, label_int].item()
        deletion_probs.append(pd)
        insertion_probs.append(pi)

    deletion_auc = float(np.trapezoid(deletion_probs) / max(1, len(deletion_probs) - 1))
    insertion_auc = float(np.trapezoid(insertion_probs) / max(1, len(insertion_probs) - 1))
    return deletion_auc, insertion_auc, deletion_probs, insertion_probs


# ============================================================
# 9. Training, checkpointing, and recovery
# ============================================================

def save_checkpoint(path, model, optimizer, epoch, best_acc, train_ds, cfg, history=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "epoch": epoch,
        "best_acc": best_acc,
        "class_to_idx": train_ds.class_to_idx,
        "cfg": cfg.__dict__,
        "history": history or [],
    }, path)
    print("Checkpoint sauvegardé:", path)


def find_available_checkpoint(output_dir="./xai_outputs_v4"):
    candidates = [
        os.path.join(output_dir, "best_enhanced_xai_model.pt"),
        os.path.join(output_dir, "last_checkpoint.pt"),
        os.path.join(output_dir, "batch_safety_checkpoint.pt"),
    ]
    for p in candidates:
        if os.path.exists(p):
            print("Checkpoint utilisé:", p)
            return p
    raise FileNotFoundError(f"Aucun checkpoint trouvé dans {output_dir}")


def train_one_epoch(model, loader, optimizer, cfg, epoch, best_acc, train_ds, history):
    model.train()
    logs, accs = [], []

    for step, batch in enumerate(tqdm(loader, desc="train")):
        video = batch["video"].to(cfg.device)
        labels = batch["label"].to(cfg.device)

        outputs = model(video, return_explanations=True)
        loss, parts = mixed_loss(outputs, labels, video, model, cfg)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        logs.append(parts)
        accs.append(accuracy(outputs["logits"].detach(), labels))

        if (step + 1) % 500 == 0:
            save_checkpoint(
                os.path.join(cfg.output_dir, "batch_safety_checkpoint.pt"),
                model, optimizer, epoch, best_acc, train_ds, cfg, history
            )

    mean_logs = {k: float(np.mean([d[k] for d in logs])) for k in logs[0].keys()}
    mean_logs["acc"] = float(np.mean(accs))
    return mean_logs


@torch.no_grad()
def validate(model, loader, cfg):
    model.eval()
    losses, accs = [], []
    for batch in tqdm(loader, desc="valid"):
        video = batch["video"].to(cfg.device)
        labels = batch["label"].to(cfg.device)
        logits = model(video, return_explanations=False)
        losses.append(float(F.cross_entropy(logits, labels).cpu()))
        accs.append(accuracy(logits, labels))
    return {"val_loss": float(np.mean(losses)), "val_acc": float(np.mean(accs))}


def train_main(resume_from: Optional[str] = None):
    cfg = CFG()
    seed_everything(cfg.seed)
    os.makedirs(cfg.output_dir, exist_ok=True)

    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = build_loaders(cfg)

    model = EnhancedAttentionGatedTransformerGradCAMModel(cfg, num_classes=cfg.num_classes).to(cfg.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    start_epoch = 0
    best_acc = -1.0
    history = []

    if resume_from is not None and os.path.exists(resume_from):
        ckpt = torch.load(resume_from, map_location=cfg.device)
        model.load_state_dict(ckpt["model"], strict=True)
        if ckpt.get("optimizer") is not None:
            optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = int(ckpt.get("epoch", -1)) + 1
        best_acc = float(ckpt.get("best_acc", -1.0))
        history = ckpt.get("history", [])
        print(f"Reprise depuis epoch index {start_epoch}. Best val_acc précédent: {best_acc:.4f}")

    for epoch in range(start_epoch, cfg.epochs):
        tr = train_one_epoch(model, train_loader, optimizer, cfg, epoch, best_acc, train_ds, history)
        va = validate(model, val_loader, cfg)

        row = {"epoch": epoch + 1, **tr, **va}
        history.append(row)
        print(f"Epoch {epoch+1}/{cfg.epochs} | train={tr} | valid={va}")

        save_checkpoint(os.path.join(cfg.output_dir, "last_checkpoint.pt"), model, optimizer, epoch, best_acc, train_ds, cfg, history)

        if va["val_acc"] > best_acc:
            best_acc = va["val_acc"]
            save_checkpoint(os.path.join(cfg.output_dir, "best_enhanced_xai_model.pt"), model, optimizer, epoch, best_acc, train_ds, cfg, history)

        pd.DataFrame(history).to_csv(os.path.join(cfg.output_dir, "training_history.csv"), index=False)

    return model, optimizer, cfg


# ============================================================
# 10. Single-video explanation and full test evaluation
# ============================================================

def load_model_from_checkpoint(checkpoint_path: str):
    cfg = CFG()
    ckpt = torch.load(checkpoint_path, map_location=cfg.device)
    if "cfg" in ckpt:
        for k, v in ckpt["cfg"].items():
            if hasattr(cfg, k):
                setattr(cfg, k, v)
    cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

    model = EnhancedAttentionGatedTransformerGradCAMModel(cfg, num_classes=cfg.num_classes).to(cfg.device)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    class_to_idx = ckpt.get("class_to_idx")
    return model, cfg, class_to_idx


def get_dataset_for_split(cfg: CFG, split: str, class_to_idx=None):
    return UCF101VideoFolderDataset(
        root_dir=split_dir(cfg, split),
        clip_len=cfg.clip_len,
        frame_size=cfg.frame_size,
        train=False,
        class_to_idx=class_to_idx,
    )


def explain_one_sample(checkpoint_path: str, sample_index: int = 0, split: str = "test", output_dir: Optional[str] = None):
    model, cfg, class_to_idx = load_model_from_checkpoint(checkpoint_path)
    dataset = get_dataset_for_split(cfg, split, class_to_idx)
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    output_dir = output_dir or cfg.output_dir
    os.makedirs(output_dir, exist_ok=True)

    sample = dataset[sample_index]
    video = sample["video"].unsqueeze(0).to(cfg.device)
    label = sample["label"].to(cfg.device)
    target_class = int(label.item())

    gradcam = VideoGradCAM(model)
    heatmap, alpha, beta, pred_idx, prob = gradcam(video)
    pred_text = idx_to_class.get(pred_idx, str(pred_idx))

    temporal_scores = temporal_importance_from_heatmap(heatmap)
    dynamics = describe_transformer_dynamics(heatmap, sample["video"], cfg)

    save_path = os.path.join(output_dir, f"explanation_{split}_{sample_index}_{pred_text}.mp4")
    save_explanation_video(sample["video"], heatmap, temporal_scores, pred_text, prob, dynamics, save_path)

    st_del_auc, st_ins_auc, st_drop, st_gain, st_del_curve, st_ins_curve = spatiotemporal_deletion_insertion_score(
        model, video, target_class, heatmap, steps=getattr(cfg, "eval_steps", 8)
    )
    fr_del_auc, fr_ins_auc, fr_del_curve, fr_ins_curve = temporal_frame_deletion_insertion_score(
        model, video, target_class, temporal_scores, steps=getattr(cfg, "eval_steps", 8)
    )
    stability = heatmap_temporal_stability(heatmap)
    motion_proxy = localization_motion_proxy(heatmap, sample["video"], cfg)
    temp_metrics = compute_temporal_explanation_metrics(heatmap)
    gate_metrics = gate_diagnostics(alpha, beta)

    report = {
        "split": split,
        "sample_index": sample_index,
        "video_path": sample["path"],
        "ground_truth": sample["class_name"],
        "prediction": pred_text,
        "correct": bool(pred_text == sample["class_name"]),
        "probability": float(prob),
        "important_dynamics": dynamics,
        "spatiotemporal_deletion_auc_lower_is_better": float(st_del_auc),
        "spatiotemporal_insertion_auc_higher_is_better": float(st_ins_auc),
        "spatiotemporal_deletion_drop_higher_is_better": float(st_drop),
        "spatiotemporal_insertion_gain_higher_is_better": float(st_gain),
        "temporal_deletion_auc_frame_level_lower_is_better": float(fr_del_auc),
        "temporal_insertion_auc_frame_level_higher_is_better": float(fr_ins_auc),
        "temporal_stability_lower_is_better": float(stability),
        "motion_localization_proxy_higher_is_better": float(motion_proxy),
        **temp_metrics,
        **gate_metrics,
        "output_video": save_path,
        "deletion_curve": json.dumps(st_del_curve),
        "insertion_curve": json.dumps(st_ins_curve),
    }

    print(json.dumps({k: v for k, v in report.items() if k not in ["deletion_curve", "insertion_curve"]}, indent=2, ensure_ascii=False))
    return report, sample


def evaluate_full_split(
    checkpoint_path: str,
    split: str = "test",
    output_dir: Optional[str] = None,
    max_samples: Optional[int] = None,
    save_every: int = 50,
):
    model, cfg, class_to_idx = load_model_from_checkpoint(checkpoint_path)
    dataset = get_dataset_for_split(cfg, split, class_to_idx)
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    output_dir = output_dir or os.path.join(cfg.output_dir, "full_split_evaluation")
    os.makedirs(output_dir, exist_ok=True)

    print("Checkpoint chargé:", checkpoint_path)
    print(f"Backbone: {getattr(cfg, 'backbone_name', 's3d')} | temporal_module: transformer")
    print(f"clip_len: {cfg.clip_len} | frame_size: {cfg.frame_size}")
    print("Split évalué:", split)
    print("Nombre de vidéos:", len(dataset))
    print("Dossier de sortie:", output_dir)

    gradcam = VideoGradCAM(model)
    n = len(dataset) if max_samples is None else min(max_samples, len(dataset))
    rows = []

    for i in tqdm(range(n), desc=f"Evaluation {split}"):
        try:
            sample = dataset[i]
            video = sample["video"].unsqueeze(0).to(cfg.device)
            label = sample["label"].to(cfg.device)
            target_class = int(label.item())

            # Accuracy propre, sans backward Grad-CAM
            with torch.no_grad():
                out = model(video, return_explanations=False)
                if isinstance(out, dict):
                    out = out.get("logits", out)
                probs = torch.softmax(out, dim=1)
                pred_idx = int(probs.argmax(dim=1).item())
                prob = float(probs[0, pred_idx].detach().cpu())

            pred_text = idx_to_class.get(pred_idx, str(pred_idx))
            heatmap, alpha, beta, _, _ = gradcam(video, class_idx=pred_idx)
            temporal_scores = temporal_importance_from_heatmap(heatmap)

            st_del_auc, st_ins_auc, st_drop, st_gain, _, _ = spatiotemporal_deletion_insertion_score(
                model, video, target_class, heatmap, steps=getattr(cfg, "eval_steps", 8)
            )
            fr_del_auc, fr_ins_auc, _, _ = temporal_frame_deletion_insertion_score(
                model, video, target_class, temporal_scores, steps=getattr(cfg, "eval_steps", 8)
            )
            stability = heatmap_temporal_stability(heatmap)
            motion_proxy = localization_motion_proxy(heatmap, sample["video"], cfg)
            temp_metrics = compute_temporal_explanation_metrics(heatmap)
            gate_metrics = gate_diagnostics(alpha, beta)
            dyn = describe_transformer_dynamics(heatmap, sample["video"], cfg)

            rows.append({
                "index": i,
                "video_path": sample["path"],
                "ground_truth": sample["class_name"],
                "prediction": pred_text,
                "correct": bool(pred_text == sample["class_name"]),
                "probability": prob,
                "spatiotemporal_deletion_auc_lower_is_better": float(st_del_auc),
                "spatiotemporal_insertion_auc_higher_is_better": float(st_ins_auc),
                "spatiotemporal_deletion_drop_higher_is_better": float(st_drop),
                "spatiotemporal_insertion_gain_higher_is_better": float(st_gain),
                "temporal_deletion_auc_frame_level_lower_is_better": float(fr_del_auc),
                "temporal_insertion_auc_frame_level_higher_is_better": float(fr_ins_auc),
                "temporal_stability_lower_is_better": float(stability),
                "motion_localization_proxy_higher_is_better": float(motion_proxy),
                "dynamic_motion_intensity_value": dyn.get("motion_intensity_value"),
                "dynamic_localization_value": dyn.get("localization_value"),
                "dynamic_variation_value": dyn.get("variation_value"),
                **temp_metrics,
                **gate_metrics,
                "error": None,
            })
        except Exception as e:
            rows.append({"index": i, "error": str(e)})

        if save_every and (i + 1) % save_every == 0:
            pd.DataFrame(rows).to_csv(os.path.join(output_dir, f"{split}_metrics_partial.csv"), index=False)

    df = pd.DataFrame(rows)
    csv_path = os.path.join(output_dir, f"{split}_metrics.csv")
    df.to_csv(csv_path, index=False)
    print("Métriques sauvegardées:", csv_path)
    return df



def summarize_results(df: pd.DataFrame, output_dir: Optional[str] = None):
    output_dir = output_dir or "./xai_outputs_transformer_s3d/full_split_evaluation"
    os.makedirs(output_dir, exist_ok=True)
    df_ok = df[df["error"].isna()].copy() if "error" in df.columns else df.copy()

    metrics = [
        "probability",
        "spatiotemporal_deletion_auc_lower_is_better",
        "spatiotemporal_insertion_auc_higher_is_better",
        "spatiotemporal_deletion_drop_higher_is_better",
        "temporal_deletion_auc_frame_level_lower_is_better",
        "temporal_insertion_auc_frame_level_higher_is_better",
        "temporal_stability_lower_is_better",
        "motion_localization_proxy_higher_is_better",
        "temporal_explanation_entropy_lower_is_more_selective",
        "temporal_explanation_concentration_higher_is_more_selective",
        "temporal_explanation_top80_frames_lower_is_more_selective",
        "temporal_explanation_coverage_ratio",
        "temporal_gate_alpha_entropy_diagnostic",
        "temporal_gate_alpha_concentration_diagnostic",
        "temporal_gate_beta_mean_diagnostic",
        "temporal_gate_beta_std_diagnostic",
        "dynamic_motion_intensity_value",
        "dynamic_localization_value",
        "dynamic_variation_value",
    ]

    summary = {
        "num_samples_total": len(df),
        "num_samples_valid": len(df_ok),
        "accuracy": float(df_ok["correct"].mean()),
    }
    for m in metrics:
        if m in df_ok.columns:
            summary[f"{m}_mean"] = float(pd.to_numeric(df_ok[m], errors="coerce").mean())
            summary[f"{m}_std"] = float(pd.to_numeric(df_ok[m], errors="coerce").std())
            summary[f"{m}_median"] = float(pd.to_numeric(df_ok[m], errors="coerce").median())

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(os.path.join(output_dir, "summary_global_metrics.csv"), index=False)

    by_class = df_ok.groupby("ground_truth").agg(
        n=("ground_truth", "count"),
        accuracy=("correct", "mean"),
        confidence_mean=("probability", "mean"),
        deletion_auc_mean=("spatiotemporal_deletion_auc_lower_is_better", "mean"),
        insertion_auc_mean=("spatiotemporal_insertion_auc_higher_is_better", "mean"),
        deletion_drop_mean=("spatiotemporal_deletion_drop_higher_is_better", "mean"),
        stability_mean=("temporal_stability_lower_is_better", "mean"),
        motion_proxy_mean=("motion_localization_proxy_higher_is_better", "mean"),
    ).reset_index().sort_values("accuracy")
    by_class.to_csv(os.path.join(output_dir, "summary_by_class.csv"), index=False)

    print("Diagnostic global")
    print("-----------------")
    print(f"Accuracy globale: {summary['accuracy']:.4f}")
    print(f"Deletion AUC spatio-temporelle: {summary.get('spatiotemporal_deletion_auc_lower_is_better_mean', np.nan):.4f} | plus bas = meilleur")
    print(f"Insertion AUC spatio-temporelle: {summary.get('spatiotemporal_insertion_auc_higher_is_better_mean', np.nan):.4f} | plus haut = meilleur")
    print(f"Deletion drop final: {summary.get('spatiotemporal_deletion_drop_higher_is_better_mean', np.nan):.4f} | plus haut = plus causal")
    print(f"Deletion AUC temporelle frame-level: {summary.get('temporal_deletion_auc_frame_level_lower_is_better_mean', np.nan):.4f} | diagnostic secondaire")
    print(f"Insertion AUC temporelle frame-level: {summary.get('temporal_insertion_auc_frame_level_higher_is_better_mean', np.nan):.4f} | diagnostic secondaire")
    print(f"Stabilité moyenne: {summary.get('temporal_stability_lower_is_better_mean', np.nan):.6f} | plus bas = meilleur")
    print(f"Motion proxy moyen: {summary.get('motion_localization_proxy_higher_is_better_mean', np.nan):.4f} | plus haut = meilleur")

    print("\nAxes complémentaires")
    print("--------------------")
    print(f"Entropie explication temporelle: {summary.get('temporal_explanation_entropy_lower_is_more_selective_mean', np.nan):.4f} | plus bas = plus sélectif")
    print(f"Concentration explication temporelle: {summary.get('temporal_explanation_concentration_higher_is_more_selective_mean', np.nan):.4f} | plus haut = plus sélectif")
    print(f"Frames nécessaires pour 80% de l'importance: {summary.get('temporal_explanation_top80_frames_lower_is_more_selective_mean', np.nan):.2f} | plus bas = plus concentré")
    print(f"Couverture temporelle explicative moyenne: {summary.get('temporal_explanation_coverage_ratio_mean', np.nan):.4f}")

    print("\nDiagnostic porte Transformer")
    print("----------------------------")
    print(f"Entropie alpha gate: {summary.get('temporal_gate_alpha_entropy_diagnostic_mean', np.nan):.4f}")
    print(f"Concentration alpha gate: {summary.get('temporal_gate_alpha_concentration_diagnostic_mean', np.nan):.4f}")
    print(f"Moyenne beta gate: {summary.get('temporal_gate_beta_mean_diagnostic_mean', np.nan):.4f}")
    print(f"Écart-type beta gate: {summary.get('temporal_gate_beta_std_diagnostic_mean', np.nan):.4f}")

    print("\nAxe dynamique")
    print("-------------")
    print(f"Intensité dynamique moyenne: {summary.get('dynamic_motion_intensity_value_mean', np.nan):.4f}")
    print(f"Localisation dynamique moyenne: {summary.get('dynamic_localization_value_mean', np.nan):.2f}%")
    print(f"Variation dynamique moyenne: {summary.get('dynamic_variation_value_mean', np.nan):.4f}")

    return summary_df, by_class

def export_outputs(output_dir="./xai_outputs_v4"):
    zip_path = "/kaggle/working/xai_outputs_v4_export"
    shutil.make_archive(zip_path, "zip", output_dir)
    print("Archive prête:", zip_path + ".zip")




## 1. Entraînement 

In [ ]:
# Nouveau training
model, optimizer, cfg = train_main()

Train videos: 10055 | Val videos: 1673 | Test videos: 1723
Classes: 101
Downloading: "https://download.pytorch.org/models/s3d-d76dad2f.pth" to /root/.cache/torch/hub/checkpoints/s3d-d76dad2f.pth


100%|██████████| 32.0M/32.0M [00:00<00:00, 190MB/s]
/tmp/ipykernel_58/2014569602.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
train:  10%|▉         | 500/5028 [10:30<1:41:14,  1.34s/it]

Checkpoint sauvegardé: ./xai_outputs_v4/batch_safety_checkpoint.pt


train:  20%|█▉        | 1000/5028 [21:03<1:32:52,  1.38s/it]

Checkpoint sauvegardé: ./xai_outputs_v4/batch_safety_checkpoint.pt


...

valid: 100%|██████████| 837/837 [02:15<00:00,  6.17it/s]


Epoch 6/10 | train={'loss_total': 1.1467987852098454, 'loss_pred': 1.0015357598417602, 'spatial_sparse': 0.2710075071978019, 'spatial_tv': 0.3902131898377562, 'temporal_entropy': 0.9995597207958772, 'temporal_smooth': 0.010428106661931542, 'temporal_coverage': 0.00045113417045261996, 'motion_align_optical_flow': 0.006895522343732836, 'spatial_focus_iou_loss': 0.820754218736905, 'input_fidelity': 0.0751962662171039, 'input_insertion_bce': 0.04308436203383275, 'input_deletion_bce': 0.010371294452454504, 'strong_causal': 0.08696243885605223, 'acc': 0.772673031026253} | valid={'val_loss': 2.639940803901813, 'val_acc': 0.6039426523297491}
Checkpoint sauvegardé: ./xai_outputs_v4/last_checkpoint.pt


train:  10%|▉         | 500/5028 [10:22<1:43:18,  1.37s/it]

Checkpoint sauvegardé: ./xai_outputs_v4/batch_safety_checkpoint.pt


train:  13%|█▎        | 637/5028 [13:12<1:30:25,  1.24s/it]

In [6]:
# Reprise 
model, optimizer, cfg = train_main(resume_from="/kaggle/input/models/younessouarda/transformer-cehckpt/pytorch/default/1/last_checkpoint (25).pt")

Train videos: 10055 | Val videos: 1673 | Test videos: 1723
Classes: 101
Downloading: "https://download.pytorch.org/models/s3d-d76dad2f.pth" to /root/.cache/torch/hub/checkpoints/s3d-d76dad2f.pth


100%|██████████| 32.0M/32.0M [00:00<00:00, 162MB/s] 
/tmp/ipykernel_58/2014569602.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


Reprise depuis epoch index 6. Best val_acc précédent: 0.6691


train:  10%|▉         | 500/5028 [10:42<1:45:47,  1.40s/it]

Checkpoint sauvegardé: ./xai_outputs_v4/batch_safety_checkpoint.pt


...

Checkpoint sauvegardé: ./xai_outputs_v4/batch_safety_checkpoint.pt


train:  99%|█████████▉| 5000/5028 [1:48:00<00:39,  1.42s/it]

Checkpoint sauvegardé: ./xai_outputs_v4/batch_safety_checkpoint.pt


valid: 100%|██████████| 837/837 [02:15<00:00,  6.16it/s]


Epoch 10/10 | train={'loss_total': 0.7670891159077011, 'loss_pred': 0.6307173806672145, 'spatial_sparse': 0.24555161216591878, 'spatial_tv': 0.3719352275879969, 'temporal_entropy': 0.9998212154886508, 'temporal_smooth': 0.0045358805510270755, 'temporal_coverage': 0.00026687465144899247, 'motion_align_optical_flow': 0.007066998898518208, 'spatial_focus_iou_loss': 0.8291508044244377, 'input_fidelity': 0.050945346209574555, 'input_insertion_bce': 0.032609472244014955, 'input_deletion_bce': 0.005627665441707009, 'strong_causal': 0.050832833899162835, 'acc': 0.8653540175019888} | valid={'val_loss': 2.244367260120825, 'val_acc': 0.6863799283154122}
Checkpoint sauvegardé: ./xai_outputs_v4/last_checkpoint.pt


## 2. Récupérer le checkpoint

In [8]:
checkpoint_path = find_available_checkpoint("./xai_outputs_v4")
checkpoint_path

Checkpoint utilisé: ./xai_outputs_v4/best_enhanced_xai_model.pt


'./xai_outputs_v4/best_enhanced_xai_model.pt'

## 3. Expliquer une vidéo test

In [9]:
report, sample = explain_one_sample(
    checkpoint_path=checkpoint_path,
    sample_index=0,
    split="test"
)

show_original_and_explanation(report, sample, width=750)

/tmp/ipykernel_58/2014569602.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


{
  "split": "test",
  "sample_index": 0,
  "video_path": "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/ApplyEyeMakeup/v_ApplyEyeMakeup_g01_c02.avi",
  "ground_truth": "ApplyEyeMakeup",
  "prediction": "ApplyEyeMakeup",
  "correct": true,
  "probability": 0.9999980926513672,
  "important_dynamics": {
    "segment": "frames 0 à 31",
    "motion_intensity_label": "faible",
    "motion_intensity_value": 0.0297,
    "localization_label": "localisé",
    "localization_value": 75.0,
    "variation_label": "faible",
    "variation_value": 0.0061,
    "continuity": "stable",
    "peak_frame": 28
  },
  "spatiotemporal_deletion_auc_lower_is_better": 0.3169222783016039,
  "spatiotemporal_insertion_auc_higher_is_better": 0.7207923015412234,
  "spatiotemporal_deletion_drop_higher_is_better": 0.9999697026887588,
  "spatiotemporal_insertion_gain_higher_is_better": 0.9996681028860621,
  "temporal_deletion_auc_frame_level_lower_is_better": 0.6965618250342231,
  "temporal_inserti

Vidéo expliquée


In [10]:
report, sample = explain_one_sample(
    checkpoint_path=checkpoint_path,
    sample_index=1,
    split="test"
)

show_original_and_explanation(report, sample, width=750)

/tmp/ipykernel_58/2014569602.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


{
  "split": "test",
  "sample_index": 1,
  "video_path": "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/ApplyEyeMakeup/v_ApplyEyeMakeup_g01_c04.avi",
  "ground_truth": "ApplyEyeMakeup",
  "prediction": "BlowDryHair",
  "correct": false,
  "probability": 0.9424101114273071,
  "important_dynamics": {
    "segment": "frames 11 à 18",
    "motion_intensity_label": "moyen",
    "motion_intensity_value": 0.0368,
    "localization_label": "localisé",
    "localization_value": 75.0,
    "variation_label": "faible",
    "variation_value": 0.0091,
    "continuity": "stable",
    "peak_frame": 12
  },
  "spatiotemporal_deletion_auc_lower_is_better": 0.15345314052046888,
  "spatiotemporal_insertion_auc_higher_is_better": 0.41547181853638904,
  "spatiotemporal_deletion_drop_higher_is_better": 0.049788032221840695,
  "spatiotemporal_insertion_gain_higher_is_better": 0.001960801804671064,
  "temporal_deletion_auc_frame_level_lower_is_better": 0.2711269289120537,
  "temporal_ins

Vidéo expliquée


In [11]:
report, sample = explain_one_sample(
    checkpoint_path=checkpoint_path,
    sample_index=310,
    split="test"
)

show_original_and_explanation(report, sample, width=750)

/tmp/ipykernel_58/2014569602.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


{
  "split": "test",
  "sample_index": 310,
  "video_path": "/kaggle/input/datasets/matthewjansen/ucf101-action-recognition/test/BoxingPunchingBag/v_BoxingPunchingBag_g22_c05.avi",
  "ground_truth": "BoxingPunchingBag",
  "prediction": "BoxingPunchingBag",
  "correct": true,
  "probability": 1.0,
  "important_dynamics": {
    "segment": "frames 16 à 23",
    "motion_intensity_label": "faible",
    "motion_intensity_value": 0.0283,
    "localization_label": "localisé",
    "localization_value": 75.0,
    "variation_label": "faible",
    "variation_value": 0.0061,
    "continuity": "stable",
    "peak_frame": 19
  },
  "spatiotemporal_deletion_auc_lower_is_better": 1.0,
  "spatiotemporal_insertion_auc_higher_is_better": 0.9381055502453819,
  "spatiotemporal_deletion_drop_higher_is_better": 0.0,
  "spatiotemporal_insertion_gain_higher_is_better": 0.9903111960738897,
  "temporal_deletion_auc_frame_level_lower_is_better": 0.8624128057572307,
  "temporal_insertion_auc_frame_level_higher_is_b

Vidéo expliquée


##

## 4. Évaluation complète du split test

In [12]:
df_test = evaluate_full_split(checkpoint_path, split="test", max_samples=None)
summary_df, class_summary = summarize_results(df_test)

/tmp/ipykernel_58/2014569602.py:59: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


Checkpoint chargé: ./xai_outputs_v4/best_enhanced_xai_model.pt
Backbone: s3d | temporal_module: transformer
clip_len: 32 | frame_size: 224
Split évalué: test
Nombre de vidéos: 1723
Dossier de sortie: ./xai_outputs_v4/full_split_evaluation


Evaluation test: 100%|██████████| 1723/1723 [1:02:31<00:00,  2.18s/it]

Métriques sauvegardées: ./xai_outputs_v4/full_split_evaluation/test_metrics.csv
Diagnostic global
-----------------
Accuracy globale: 0.7290
Deletion AUC spatio-temporelle: 0.3860 | plus bas = meilleur
Insertion AUC spatio-temporelle: 0.4662 | plus haut = meilleur
Deletion drop final: 0.5593 | plus haut = plus causal
Deletion AUC temporelle frame-level: 0.5077 | diagnostic secondaire
Insertion AUC temporelle frame-level: 0.5542 | diagnostic secondaire
Stabilité moyenne: 0.003384 | plus bas = meilleur
Motion proxy moyen: 0.2680 | plus haut = meilleur

Axes complémentaires
--------------------
Entropie explication temporelle: 0.9017 | plus bas = plus sélectif
Concentration explication temporelle: 0.0494 | plus haut = plus sélectif
Frames nécessaires pour 80% de l'importance: 16.01 | plus bas = plus concentré
Couverture temporelle explicative moyenne: 0.2561

Diagnostic porte Transformer
----------------------------
Entropie alpha gate: 0.9967
Concentration alpha gate: 0.2521
Moyenne beta